# LatentMT × AlexandriaX — complete Ouro-2.6B experiment

**One launch, one kernel, one isolated run directory.** This notebook:

1. trains only on the frozen **66,480 original TRAIN + 12,250 official DEV + 8,670 saved public-60%** turns;
2. keeps the saved **5,772-turn locked-40%** split out of training and retrieval;
3. fine-tunes [ByteDance/Ouro-2.6B-Thinking](https://huggingface.co/ByteDance/Ouro-2.6B-Thinking) with its fixed four latent recurrent passes, LoRA, bf16, context-3, and deterministic two-shot country/domain prompting;
4. resumes safely from the latest 100-step checkpoint;
5. evaluates the selected adapter on locked-40% with official per-country spBLEU and chrF++;
6. runs resumable Beam-4 inference on the 14,459-turn private test and emits a validated ZIP containing only `predictions.jsonl`.

The method follows the recurrent-depth idea in [LatentMT (arXiv:2607.18618)](https://arxiv.org/abs/2607.18618). The notebook **never rebuilds the public 60/40 split**: it discovers the already-saved artifacts, checks exact sizes/schemas/conversation disjointness, and stops on ambiguity.

Run top-to-bottom in the existing `axmt_py311` kernel. To start a genuinely new experiment, change only `RUN_NAME`; rerunning the same name resumes it.


In [2]:
# Cell 1 — Verify kernel and install dependencies

import os
import sys
from pathlib import Path

AXMT_HOME = Path(
    os.environ.get(
        "AXMT_HOME",
        "/home/mabdallah/alexandriax_mt_14d",
    )
).expanduser()

EXPECTED_ENV = (AXMT_HOME / "envs" / "axmt_py311").resolve()
CURRENT_ENV = Path(sys.prefix).resolve()

if CURRENT_ENV != EXPECTED_ENV:
    raise RuntimeError(
        "Wrong Jupyter kernel.\n\n"
        f"Current Python: {sys.executable}\n"
        f"Current environment: {CURRENT_ENV}\n"
        f"Required environment: {EXPECTED_ENV}\n\n"
        "Select: Kernel → Change Kernel → axmt_py311, "
        "restart the kernel, then rerun this cell."
    )

print("Correct kernel:", sys.executable)

%pip install -q -U "transformers>=4.56,<5" "peft>=0.17,<1" "accelerate>=1.7,<2" "datasets>=3,<5" "sacrebleu>=2.5,<3" "huggingface_hub>=0.34,<1" "sentencepiece>=0.2" "safetensors>=0.5" "pandas>=2,<3" "pyarrow>=18" "tqdm>=4.67" "einops>=0.8"

import torch
import transformers
import peft
import datasets
import sacrebleu
import pandas as pd

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("SacreBLEU:", sacrebleu.__version__)

assert torch.cuda.is_available(), "CUDA GPU is unavailable."
assert torch.cuda.is_bf16_supported(), "The experiment requires bf16 support."

print("GPU:", torch.cuda.get_device_name(0))
print("bf16 support: yes")

Correct kernel: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/bin/python
Note: you may need to restart the kernel to use updated packages.
Python: 3.11.15
Torch: 2.11.0+cu128 | CUDA: 12.8
Transformers: 4.57.6
PEFT: 0.19.1
Datasets: 4.8.5
SacreBLEU: 2.6.0
GPU: NVIDIA GeForce RTX 5090
bf16 support: yes


In [1]:
# Frozen experiment configuration and isolated/resumable directories.
from __future__ import annotations
import os, gc, re, json, math, time, random, hashlib, shutil, zipfile
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path(os.environ.get("AXMT_HOME", "/home/mabdallah/alexandriax_mt_14d")).expanduser().resolve()
RUN_NAME = "ouro26b_latentmt_u4_full87400_ctx3_2shot_r32a64_lr2e4_2ep_bf16_v1"
RUN_ROOT = PROJECT / "latentmt_ouro_experiments" / RUN_NAME
DIRS = {
    "prepared": RUN_ROOT / "prepared",
    "checkpoints": RUN_ROOT / "checkpoints",
    "final_adapter": RUN_ROOT / "final_adapter",
    "locked": RUN_ROOT / "locked40_eval",
    "test": RUN_ROOT / "private_test_submission",
    "logs": RUN_ROOT / "logs",
}
assert PROJECT.is_dir(), f"AXMT project directory not found: {PROJECT}"
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

SEED = 3407
COUNTRY_CODES = ["EG", "JO", "LB", "LY", "MA", "MR", "OM", "PS", "SA", "SD", "SY", "TN", "YE"]
EXPECTED = {
    "original_train_turns": 66_480,
    "official_dev_turns": 12_250,
    "public60_turns": 8_670,
    "locked40_turns": 5_772,
    "training_turns": 87_400,
    "private_test_turns": 14_459,
    "private_test_conversations": 4_673,
    "countries": 13,
}
BASE_MODEL_REPO = "ByteDance/Ouro-2.6B-Thinking"
MODEL_DIR = PROJECT / "models" / "hf" / "ByteDance--Ouro-2.6B-Thinking"

MAX_CONTEXT_TURNS = 3
N_FEW_SHOTS = 2
MAX_FEWSHOT_CHARS = 450
MAX_SEQ_LEN = 2048
MAX_NEW_TOKENS = 120
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
EPOCHS, MICRO_BATCH, GRAD_ACCUM = 2.0, 1, 8
LEARNING_RATE, WARMUP_RATIO, WEIGHT_DECAY = 2e-4, 0.03, 0.01
SAVE_STEPS, SAVE_TOTAL_LIMIT, LOGGING_STEPS = 100, 50, 10
GENERATION_SAVE_EVERY = 100

# Optional path overrides; environment variables avoid editing the notebook.
ORIGINAL_TRAIN_PATH = os.environ.get(
    "AXMT_ORIGINAL_TRAIN_PATH",
    str(PROJECT / "inference_variants/_shared_cache/paired_dev_train_v3/train_fewshot_pool_df.pkl"),
)
OFFICIAL_DEV_PATH = os.environ.get(
    "AXMT_OFFICIAL_DEV_PATH",
    str(PROJECT / "inference_variants/_shared_cache/paired_dev_train_v3/official_dev_df.pkl"),
)
PUBLIC60_PATH = os.environ.get("AXMT_PUBLIC60_PATH") or None
LOCKED40_PATH = os.environ.get("AXMT_LOCKED40_PATH") or None
PRIVATE_TEST_ARCHIVE = os.environ.get("AXMT_PRIVATE_TEST_ARCHIVE") or None
SELECTED_ADAPTER_OVERRIDE = os.environ.get("AXMT_SELECTED_ADAPTER") or None

os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

def atomic_json(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")
    os.replace(tmp, path)

frozen = {
    "run_name": RUN_NAME, "base_model": BASE_MODEL_REPO, "seed": SEED,
    "countries": COUNTRY_CODES, "expected": EXPECTED, "max_context_turns": MAX_CONTEXT_TURNS,
    "few_shots": N_FEW_SHOTS, "max_fewshot_chars": MAX_FEWSHOT_CHARS,
    "max_seq_len": MAX_SEQ_LEN, "max_new_tokens": MAX_NEW_TOKENS,
    "latent_recurrent_steps": 4,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "train": {"epochs": EPOCHS, "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
              "lr": LEARNING_RATE, "warmup_ratio": WARMUP_RATIO,
              "weight_decay": WEIGHT_DECAY, "save_steps": SAVE_STEPS},
    "decode": {"num_beams": 4, "do_sample": False, "length_penalty": 1.0,
               "repetition_penalty": 1.05, "early_stopping": True},
}
frozen_path = RUN_ROOT / "frozen_run_config.json"
if frozen_path.exists():
    old = json.loads(frozen_path.read_text(encoding="utf-8"))
    assert old == frozen, (
        f"RUN_NAME already exists with a different frozen configuration: {RUN_ROOT}\n"
        "Use a new RUN_NAME instead of mixing experiments."
    )
else:
    atomic_json(frozen, frozen_path)

print("Project:", PROJECT)
print("Run:", RUN_ROOT)
print("Effective batch:", MICRO_BATCH * GRAD_ACCUM)
print("Approx. optimizer steps:", math.ceil(EXPECTED["training_turns"] / (MICRO_BATCH * GRAD_ACCUM) * EPOCHS))


Project: /home/mabdallah/alexandriax_mt_14d
Run: /home/mabdallah/alexandriax_mt_14d/latentmt_ouro_experiments/ouro26b_latentmt_u4_full87400_ctx3_2shot_r32a64_lr2e4_2ep_bf16_v1
Effective batch: 8
Approx. optimizer steps: 21850


In [2]:
# Cell 3 — Load exact frozen data artifacts

import ast
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


DATA_RUN = (
    PROJECT
    / "data"
    / "nilechat3b_dev_continuation"
    / "nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1"
)

ORIGINAL_TRAIN_FILE = (
    PROJECT
    / "inference_variants"
    / "_shared_cache"
    / "paired_dev_train_v3"
    / "train_fewshot_pool_df.pkl"
)

CONTINUATION_FILE = (
    DATA_RUN
    / "fine_tune_rows_dev_plus_public60.pkl"
)

LOCKED40_FILE = (
    DATA_RUN
    / "selection_rows_public40.pkl"
)

ASSIGNMENTS_FILE = (
    DATA_RUN
    / "public_test_grouped_60_40_assignments.csv"
)

for path in [
    ORIGINAL_TRAIN_FILE,
    CONTINUATION_FILE,
    LOCKED40_FILE,
]:
    assert path.exists(), f"Missing required artifact: {path}"

print("Data folder:", DATA_RUN)


COUNTRY_ALIASES = {
    "EGYPT": "EG",
    "EGYPTIAN": "EG",
    "JORDAN": "JO",
    "JORDANIAN": "JO",
    "LEBANON": "LB",
    "LEBANESE": "LB",
    "LIBYA": "LY",
    "LIBYAN": "LY",
    "MOROCCO": "MA",
    "MOROCCAN": "MA",
    "MAURITANIA": "MR",
    "MAURITANIAN": "MR",
    "OMAN": "OM",
    "OMANI": "OM",
    "PALESTINE": "PS",
    "PALESTINIAN": "PS",
    "SAUDI": "SA",
    "SAUDIARABIA": "SA",
    "SUDAN": "SD",
    "SUDANESE": "SD",
    "SYRIA": "SY",
    "SYRIAN": "SY",
    "TUNISIA": "TN",
    "TUNISIAN": "TN",
    "YEMEN": "YE",
    "YEMENI": "YE",
}


def clean_scalar(value):
    if value is None:
        return ""

    try:
        missing = pd.isna(value)

        if isinstance(missing, (bool, np.bool_)) and missing:
            return ""
    except Exception:
        pass

    return str(value).strip()


def normalize_country(value):
    raw = clean_scalar(value).upper()
    raw = "".join(
        character
        for character in raw
        if character.isalpha()
    )

    if raw in COUNTRY_CODES:
        return raw

    return COUNTRY_ALIASES.get(raw, raw)


def parse_list(value):
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, str) and value.strip():
        for parser in [json.loads, ast.literal_eval]:
            try:
                result = parser(value)

                if isinstance(result, list):
                    return result
            except Exception:
                pass

    return []


def get_ci(mapping, names, default=None):
    columns = {
        str(key).lower(): key
        for key in mapping.keys()
    }

    for name in names:
        key = columns.get(name.lower())

        if key is not None:
            value = mapping[key]

            if value is not None:
                return value

    return default


def turn_text(turn):
    if isinstance(turn, str):
        return turn.strip()

    if not isinstance(turn, dict):
        return clean_scalar(turn)

    return clean_scalar(
        get_ci(
            turn,
            [
                "text",
                "utterance",
                "content",
                "value",
                "english",
                "source_text",
            ],
            "",
        )
    )


def turn_value(turn, names, default=""):
    if not isinstance(turn, dict):
        return clean_scalar(default)

    return clean_scalar(
        get_ci(
            turn,
            names,
            default,
        )
    )


def turn_order(turn, fallback):
    raw = (
        get_ci(
            turn,
            [
                "turn_order",
                "turn_id",
                "order",
                "idx",
                "index",
            ],
            fallback,
        )
        if isinstance(turn, dict)
        else fallback
    )

    try:
        return int(raw)
    except Exception:
        return int(fallback)


def infer_country_from_path(path):
    for part in reversed(Path(path).parts):
        for token in part.replace("-", "_").split("_"):
            country = normalize_country(token)

            if country in COUNTRY_CODES:
                return country

    return ""


def pick_series(frame, aliases, default=""):
    lookup = {
        str(column).lower(): column
        for column in frame.columns
    }

    for alias in aliases:
        column = lookup.get(alias.lower())

        if column is not None:
            return frame[column]

    return pd.Series(
        [default] * len(frame),
        index=frame.index,
    )


def normalize_frame(
    frame,
    origin,
    expected_rows,
    expected_countries=None,
):
    frame = frame.copy().reset_index(drop=True)

    output = pd.DataFrame(index=frame.index)

    output["config"] = pick_series(
        frame,
        ["config", "country", "country_code"],
    ).map(normalize_country)

    output["conversation_id"] = pick_series(
        frame,
        [
            "conversation_id",
            "conv_id",
            "dialogue_id",
            "dialog_id",
        ],
    ).map(clean_scalar)

    raw_source_id = pick_series(
        frame,
        ["source_id", "id"],
    ).map(clean_scalar)

    missing_conversation = output[
        "conversation_id"
    ].eq("")

    if missing_conversation.any():
        parsed = (
            raw_source_id
            .str.rsplit("_", n=1)
            .str[0]
            .str.replace(
                r"^[A-Z]{2}_(train|dev|test)_",
                "",
                regex=True,
            )
        )

        output.loc[
            missing_conversation,
            "conversation_id",
        ] = parsed[missing_conversation]

    raw_order = pd.to_numeric(
        pick_series(
            frame,
            [
                "turn_order",
                "turn_id",
                "order",
                "idx",
                "index",
            ],
            np.nan,
        ),
        errors="coerce",
    )

    if raw_order.isna().any():
        parsed_order = pd.to_numeric(
            raw_source_id.str.rsplit(
                "_",
                n=1,
            ).str[-1],
            errors="coerce",
        )

        raw_order = raw_order.fillna(
            parsed_order
        )

    if raw_order.isna().any():
        fallback = (
            output.groupby(
                [
                    "config",
                    "conversation_id",
                ],
                sort=False,
            ).cumcount()
            + 1
        )

        raw_order = raw_order.fillna(
            fallback
        )

    output["turn_order"] = raw_order.astype(int)

    output["source_text"] = pick_series(
        frame,
        [
            "source_text",
            "english",
            "source",
            "input_text",
            "utterance",
        ],
    ).map(clean_scalar)

    output["target_arabic"] = pick_series(
        frame,
        [
            "target_arabic",
            "reference_arabic",
            "reference",
            "target",
            "arabic",
            "translation",
        ],
    ).map(clean_scalar)

    aliases = {
        "dialect": [
            "dialect",
            "target_dialect",
        ],
        "domain": [
            "domain",
            "topic",
        ],
        "persona": [
            "persona",
            "roles",
        ],
        "speaker": [
            "speaker",
            "role",
            "speaker_role",
            "participant",
        ],
        "gender_direction": [
            "gender_direction",
            "direction",
            "speaker_addressee_gender",
        ],
    }

    for column, names in aliases.items():
        output[column] = pick_series(
            frame,
            names,
        ).map(clean_scalar)

    generated_id = (
        output["config"]
        + "_"
        + origin
        + "_"
        + output["conversation_id"]
        + "_"
        + output["turn_order"].astype(str)
    )

    output["raw_source_id"] = (
        raw_source_id.where(
            raw_source_id.ne(""),
            generated_id,
        )
    )

    output["source_id"] = (
        origin
        + "::"
        + output["raw_source_id"]
    )

    output["_origin"] = origin

    if "previous_english_turns" in frame.columns:
        output["previous_english_turns"] = (
            frame["previous_english_turns"]
            .map(parse_list)
        )
    else:
        output["previous_english_turns"] = [
            []
            for _ in range(len(output))
        ]

    assert len(output) == expected_rows, (
        f"{origin}: expected {expected_rows:,} rows, "
        f"found {len(output):,}"
    )

    assert output["source_text"].ne("").all(), (
        f"{origin}: empty source text found"
    )

    assert output["target_arabic"].ne("").all(), (
        f"{origin}: empty target found"
    )

    assert output["conversation_id"].ne("").all(), (
        f"{origin}: missing conversation_id"
    )

    assert output["source_id"].is_unique, (
        f"{origin}: duplicate source_id"
    )

    observed_countries = sorted(
        output["config"].unique().tolist()
    )

    assert set(observed_countries).issubset(
        COUNTRY_CODES
    ), (
        f"{origin}: unexpected countries "
        f"{observed_countries}"
    )

    if expected_countries is not None:
        assert observed_countries == sorted(
            expected_countries
        ), (
            f"{origin}: countries={observed_countries}, "
            f"expected={sorted(expected_countries)}"
        )

    return output


def restore_missing_context(frame):
    frame = frame.sort_values(
        [
            "config",
            "conversation_id",
            "turn_order",
        ],
        kind="stable",
    ).reset_index(drop=True)

    has_context = frame[
        "previous_english_turns"
    ].map(len).gt(0).any()

    if has_context:
        return frame

    contexts = [
        []
        for _ in range(len(frame))
    ]

    for _, group in frame.groupby(
        [
            "config",
            "conversation_id",
        ],
        sort=False,
    ):
        indexes = group.index.tolist()

        for position, index in enumerate(indexes):
            previous = []

            for previous_index in indexes[
                max(
                    0,
                    position - MAX_CONTEXT_TURNS,
                ):position
            ]:
                row = frame.loc[previous_index]

                previous.append(
                    {
                        "turn_order": int(
                            row["turn_order"]
                        ),
                        "speaker": row["speaker"],
                        "direction": row[
                            "gender_direction"
                        ],
                        "text": row["source_text"],
                    }
                )

            contexts[index] = previous

    frame[
        "previous_english_turns"
    ] = contexts

    return frame


def frame_fingerprint(frame):
    hasher = hashlib.sha256()

    ordered = frame.sort_values(
        [
            "config",
            "raw_source_id",
        ],
        kind="stable",
    )

    for row in ordered[
        [
            "config",
            "raw_source_id",
            "source_text",
            "target_arabic",
        ]
    ].itertuples(index=False, name=None):
        hasher.update(
            json.dumps(
                row,
                ensure_ascii=False,
                separators=(",", ":"),
            ).encode("utf-8")
        )
        hasher.update(b"\n")

    return hasher.hexdigest()


original_train = normalize_frame(
    pd.read_pickle(ORIGINAL_TRAIN_FILE),
    origin="original_train",
    expected_rows=66_480,
    expected_countries=[
        "EG", "JO", "LB", "MA", "MR",
        "OM", "PS", "SA", "SY", "TN", "YE",
    ],
)

continuation_train = normalize_frame(
    pd.read_pickle(CONTINUATION_FILE),
    origin="dev_plus_public60",
    expected_rows=20_920,
    expected_countries=COUNTRY_CODES,
)

locked40 = normalize_frame(
    pd.read_pickle(LOCKED40_FILE),
    origin="locked40",
    expected_rows=5_772,
    expected_countries=COUNTRY_CODES,
)


original_train = restore_missing_context(
    original_train
)

continuation_train = restore_missing_context(
    continuation_train
)

locked40 = restore_missing_context(
    locked40
)


continuation_ids = set(
    continuation_train[
        "raw_source_id"
    ]
)

locked_ids = set(
    locked40[
        "raw_source_id"
    ]
)

assert not (
    continuation_ids
    & locked_ids
), "LOCKED LEAKAGE: common source IDs found"


def conversation_key(source_id):
    return str(source_id).rsplit(
        "_",
        1,
    )[0]


continuation_conversations = set(
    continuation_train[
        "raw_source_id"
    ].map(conversation_key)
)

locked_conversations = set(
    locked40[
        "raw_source_id"
    ].map(conversation_key)
)

assert not (
    continuation_conversations
    & locked_conversations
), "LOCKED LEAKAGE: common conversations found"


train_pool = pd.concat(
    [
        original_train,
        continuation_train,
    ],
    ignore_index=True,
)

assert len(train_pool) == 87_400
assert train_pool["source_id"].is_unique
assert sorted(
    train_pool["config"].unique().tolist()
) == COUNTRY_CODES


resolved = {
    "original_train": str(
        ORIGINAL_TRAIN_FILE
    ),
    "continuation_dev_plus_public60": str(
        CONTINUATION_FILE
    ),
    "locked40": str(
        LOCKED40_FILE
    ),
    "assignments": (
        str(ASSIGNMENTS_FILE)
        if ASSIGNMENTS_FILE.exists()
        else None
    ),
    "fingerprints": {
        "original_train": frame_fingerprint(
            original_train
        ),
        "continuation_dev_plus_public60": (
            frame_fingerprint(
                continuation_train
            )
        ),
        "locked40": frame_fingerprint(
            locked40
        ),
    },
    "train_turns": len(train_pool),
    "locked_turns": len(locked40),
    "locked_overlap": 0,
}

atomic_json(
    resolved,
    DIRS["prepared"]
    / "resolved_data.json",
)


summary = pd.DataFrame(
    [
        {
            "component": "Original TRAIN",
            "turns": len(original_train),
            "countries": original_train[
                "config"
            ].nunique(),
            "role": "TRAIN",
        },
        {
            "component": "DEV + public60",
            "turns": len(continuation_train),
            "countries": continuation_train[
                "config"
            ].nunique(),
            "role": "TRAIN",
        },
        {
            "component": "Public locked40",
            "turns": len(locked40),
            "countries": locked40[
                "config"
            ].nunique(),
            "role": "EVAL ONLY",
        },
        {
            "component": "Final training pool",
            "turns": len(train_pool),
            "countries": train_pool[
                "config"
            ].nunique(),
            "role": "TRAIN",
        },
    ]
)

display(summary)

display(
    train_pool.groupby(
        [
            "config",
            "_origin",
        ]
    ).size().unstack(
        fill_value=0
    )
)

print("Training turns:", f"{len(train_pool):,}")
print("Locked turns:", f"{len(locked40):,}")
print("Locked overlap: 0")
print("DATA VALIDATION: PASS")

Data folder: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1


,component,turns,countries,role
0,Original TRAIN,66480,11,TRAIN
1,DEV + public60,20920,13,TRAIN
2,Public locked40,5772,13,EVAL ONLY
3,Final training pool,87400,13,TRAIN


_origin,dev_plus_public60,original_train
config,,
EG,1784,3108
JO,1779,5501
LB,1782,8906
LY,665,0
MA,1778,2573
MR,1782,5515
OM,1779,6280
PS,1776,14933
SA,1778,8470


Training turns: 87,400
Locked turns: 5,772
Locked overlap: 0
DATA VALIDATION: PASS


In [3]:
# Ouro tokenizer, deterministic two-shot prompts, and cached completion-only tokenization.
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoConfig
from datasets import Dataset, load_from_disk
from tqdm.auto import tqdm

model_weight_markers = [
    MODEL_DIR / "model.safetensors",
    MODEL_DIR / "model.safetensors.index.json",
    MODEL_DIR / "pytorch_model.bin",
    MODEL_DIR / "pytorch_model.bin.index.json",
]
if not (MODEL_DIR / "config.json").exists() or not any(p.exists() for p in model_weight_markers):
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=BASE_MODEL_REPO,
        local_dir=str(MODEL_DIR),
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
    )
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

ouro_config = AutoConfig.from_pretrained(MODEL_DIR, trust_remote_code=True)
assert int(getattr(ouro_config, "total_ut_steps", -1)) == 4, (
    f"Expected Ouro fixed recurrent depth u=4, got {getattr(ouro_config, 'total_ut_steps', None)}"
)
assert float(getattr(ouro_config, "early_exit_threshold", -1)) == 1.0, (
    "LatentMT requires adaptive early exit disabled (early_exit_threshold=1.0)."
)
print("Ouro latent recurrent steps:", ouro_config.total_ut_steps)
print("Adaptive early exit:", "disabled")

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

fs_cols = [
    "source_id", "_origin", "config", "conversation_id", "dialect", "domain",
    "source_text", "target_arabic",
]
FS_RECORDS = train_pool[fs_cols].to_dict("records")  # training pool only
all_idx, short_idx = list(range(len(FS_RECORDS))), []
by_cfg, short_by_cfg, by_cfg_domain, short_by_cfg_domain = defaultdict(list), defaultdict(list), defaultdict(list), defaultdict(list)
for i, ex in enumerate(FS_RECORDS):
    key, cfg = (ex["config"], ex["domain"]), ex["config"]
    by_cfg[cfg].append(i); by_cfg_domain[key].append(i)
    if len(ex["source_text"]) + len(ex["target_arabic"]) <= MAX_FEWSHOT_CHARS:
        short_idx.append(i); short_by_cfg[cfg].append(i); short_by_cfg_domain[key].append(i)

def stable_seed(source_id):
    return int(hashlib.md5(f"{source_id}_{SEED}".encode()).hexdigest()[:8], 16)

def select_few_shots(row, n=N_FEW_SHOTS):
    if n <= 0:
        return []
    rng = random.Random(stable_seed(row["source_id"]))
    pools = [
        short_by_cfg_domain.get((row["config"], row["domain"]), []),
        short_by_cfg.get(row["config"], []), short_idx,
        by_cfg_domain.get((row["config"], row["domain"]), []),
        by_cfg.get(row["config"], []), all_idx,
    ]
    selected, selected_ids = [], set()
    for pool in pools:
        if not pool:
            continue
        # Random-index probes avoid shuffling thousands of candidates for every turn.
        attempts = min(max(16, n * 12), max(16, len(pool) * 2))
        for _ in range(attempts):
            ex = FS_RECORDS[pool[rng.randrange(len(pool))]]
            same_row = ex["source_id"] == row["source_id"]
            same_conversation = (
                ex["_origin"] == row.get("_origin", "")
                and ex["config"] == row["config"]
                and ex["conversation_id"] == row["conversation_id"]
            )
            if same_row or same_conversation or ex["source_id"] in selected_ids:
                continue
            selected.append(ex); selected_ids.add(ex["source_id"])
            if len(selected) == n:
                return selected
    raise RuntimeError(f"Could not select {n} training-only examples for {row['source_id']}")

def context_block(previous_turns, keep=MAX_CONTEXT_TURNS):
    turns = list(previous_turns or [])[-keep:] if keep else []
    if not turns:
        return "No previous context."
    lines = []
    for i, turn in enumerate(turns, 1):
        speaker, text = clean_scalar(turn.get("speaker", "")), clean_scalar(turn.get("text", ""))
        lines.append(f"{i}. {speaker}: {text}" if speaker else f"{i}. {text}")
    return "\n".join(lines)

def metadata_block(row):
    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]
    lines = [f"{name}: {clean_scalar(value)}" for name, value in fields if clean_scalar(value)]
    return "\n".join(lines) if lines else "No metadata."

def fewshot_block(shots):
    if not shots:
        return "No examples available."
    blocks = []
    for i, ex in enumerate(shots, 1):
        meta = ", ".join(x for x in [
            f"config={ex['config']}" if ex["config"] else "",
            f"dialect={ex['dialect']}" if ex["dialect"] else "",
            f"domain={ex['domain']}" if ex["domain"] else "",
        ] if x) or "no metadata"
        blocks.append(
            f"Example {i} ({meta})\nEnglish:\n{ex['source_text']}\n\nArabic:\n{ex['target_arabic']}"
        )
    return "\n\n".join(blocks)

def user_prompt(row, shots, keep_context):
    return (
        "Task:\n"
        "Translate the current English dialogue turn into the target dialectal Arabic variety.\n\n"
        "Few-shot training examples:\n"
        f"{fewshot_block(shots)}\n\n"
        "Metadata:\n"
        f"{metadata_block(row)}\n\n"
        "Previous English dialogue context:\n"
        f"{context_block(row.get('previous_english_turns', []), keep_context)}\n\n"
        "Current English turn:\n"
        f"{row['source_text']}\n\n"
        "Rules:\n"
        "- Preserve the meaning exactly.\n"
        "- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.\n"
        "- Follow the dialect/style pattern shown in the few-shot examples when relevant.\n"
        "- Do not copy the few-shot examples.\n"
        "- Preserve names, numbers, named entities, and technical terms when appropriate.\n"
        "- Keep the tone appropriate for the speaker and domain.\n"
        "- Return only the Arabic translation."
    )

def render_chat(user_text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def fitted_prompt(row, token_budget):
    shots = select_few_shots(row)
    variants = [(shots, 3), (shots, 1), (shots[:1], 1), ([], 1), ([], 0)]
    for chosen_shots, keep_context in variants:
        text = render_chat(user_prompt(row, chosen_shots, keep_context))
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) <= token_budget:
            return text, ids
    raise RuntimeError(
        f"Prompt cannot fit safely for {row['source_id']} within {token_budget} tokens; "
        "the current source turn itself is too long."
    )

def tokenize_supervised(row):
    target_ids = tokenizer(clean_scalar(row["target_arabic"]), add_special_tokens=False)["input_ids"]
    target_ids = target_ids + [tokenizer.eos_token_id]
    assert len(target_ids) < MAX_SEQ_LEN, f"Target too long: {row['source_id']}"
    _, prompt_ids = fitted_prompt(row, MAX_SEQ_LEN - len(target_ids))
    return {
        "input_ids": prompt_ids + target_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(target_ids)),
        "labels": [-100] * len(prompt_ids) + target_ids,
    }

tokenized_path = DIRS["prepared"] / "train_tokenized"
tokenized_meta_path = DIRS["prepared"] / "train_tokenized_meta.json"
expected_token_meta = {
    "train_turns": len(train_pool),
    "train_components": resolved["fingerprints"],
    "model": BASE_MODEL_REPO, "max_seq_len": MAX_SEQ_LEN,
    "context_turns": MAX_CONTEXT_TURNS, "few_shots": N_FEW_SHOTS,
}
if tokenized_path.exists():
    assert tokenized_meta_path.exists(), "Token cache exists without its integrity manifest."
    assert json.loads(tokenized_meta_path.read_text()) == expected_token_meta, (
        "Token cache/config mismatch. Use a new RUN_NAME."
    )
    train_dataset = load_from_disk(str(tokenized_path))
else:
    raw_columns = [
        "source_id", "_origin", "config", "conversation_id", "dialect", "domain",
        "speaker", "gender_direction", "source_text", "target_arabic", "previous_english_turns",
    ]
    raw_dataset = Dataset.from_pandas(train_pool[raw_columns], preserve_index=False)
    train_dataset = raw_dataset.map(
        tokenize_supervised, remove_columns=raw_dataset.column_names,
        num_proc=1, writer_batch_size=250, desc="Tokenizing completion-only training rows",
    )
    train_dataset.save_to_disk(str(tokenized_path))
    atomic_json(expected_token_meta, tokenized_meta_path)

assert len(train_dataset) == EXPECTED["training_turns"]
lengths = np.fromiter((len(x) for x in train_dataset["input_ids"]), dtype=np.int32)
supervised = np.fromiter(
    (sum(v != -100 for v in x) for x in train_dataset["labels"]), dtype=np.int32
)
assert supervised.min() > 0 and lengths.max() <= MAX_SEQ_LEN
print("Tokenized rows:", len(train_dataset))
print("Sequence lengths p50/p95/max:", *np.percentile(lengths, [50, 95, 100]).astype(int))
print("Supervised target tokens p50/p95/max:", *np.percentile(supervised, [50, 95, 100]).astype(int))
print(render_chat(user_prompt(train_pool.iloc[0].to_dict(), select_few_shots(train_pool.iloc[0].to_dict()), 3))[:1800])


Ouro latent recurrent steps: 4
Adaptive early exit: disabled
Tokenized rows: 87400
Sequence lengths p50/p95/max: 624 764 1368
Supervised target tokens p50/p95/max: 68 127 829
<|im_start|>system
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.<|im_end|>
<|im_start|>user
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
Example 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
Exactly! By the time they take their cut and we pay for transport, there's barely anything left for us.

Arabic:
بالظبط. كدة بعد ما ياخدوا مكسبهم وندفع فلوس النقل. مش هيتبقى لنا حاجة خالص.

Example 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
The last 

In [4]:
# Build recurrent Ouro + LoRA. The same shared Transformer stack is reused for all four latent passes.
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
assert int(getattr(model.config, "total_ut_steps", -1)) == 4
model.config.early_exit_threshold = 1.0
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
found_suffixes = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
missing = sorted(set(target_modules) - found_suffixes)
assert not missing, f"Ouro module names changed; missing LoRA targets: {missing}"
lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=target_modules, bias="none",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print("CUDA allocated GiB:", round(torch.cuda.memory_allocated() / 2**30, 2))


Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 60,555,264 || all params: 2,728,529,921 || trainable%: 2.2193
CUDA allocated GiB: 0.0


In [ ]:
# Train or resume from the latest 100-step checkpoint; optimizer/scheduler state is preserved.
import inspect
from dataclasses import dataclass
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

@dataclass
class CompletionOnlyCollator:
    pad_token_id: int
    def __call__(self, features):
        width = max(len(x["input_ids"]) for x in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in features:
            pad = width - len(item["input_ids"])
            batch["input_ids"].append(item["input_ids"] + [self.pad_token_id] * pad)
            batch["attention_mask"].append(item["attention_mask"] + [0] * pad)
            batch["labels"].append(item["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

train_args = TrainingArguments(
    output_dir=str(DIRS["checkpoints"]),
    overwrite_output_dir=False,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    optim="adamw_torch_fused",
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    save_safetensors=True,
    eval_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    seed=SEED,
    data_seed=SEED,
    ddp_find_unused_parameters=False,
)
trainer_kwargs = dict(
    model=model, args=train_args, train_dataset=train_dataset,
    data_collator=CompletionOnlyCollator(tokenizer.pad_token_id),
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)

completion_path = RUN_ROOT / "training_complete.json"
expected_steps = math.ceil(len(train_dataset) / (MICRO_BATCH * GRAD_ACCUM) * EPOCHS)
completion = json.loads(completion_path.read_text()) if completion_path.exists() else {}
already_complete = (
    bool(completion.get("completed"))
    and int(completion.get("global_step", -1)) >= expected_steps
    and (DIRS["final_adapter"] / "adapter_config.json").exists()
)
if already_complete:
    print(f"Training already complete at step {completion['global_step']}; preserving final adapter.")
else:
    latest = get_last_checkpoint(str(DIRS["checkpoints"])) if DIRS["checkpoints"].exists() else None
    print("Resume checkpoint:", latest or "none — starting fresh")
    train_result = trainer.train(resume_from_checkpoint=latest)
    trainer.save_model(str(DIRS["final_adapter"]))
    tokenizer.save_pretrained(str(DIRS["final_adapter"]))
    trainer.save_state()
    atomic_json(
        {
            "completed": True,
            "global_step": int(trainer.state.global_step),
            "epoch": float(trainer.state.epoch or 0),
            "train_metrics": train_result.metrics,
            "last_checkpoint_before_run": latest,
        },
        completion_path,
    )
    print("Training complete at step:", trainer.state.global_step)
print("Final adapter:", DIRS["final_adapter"])


Resume checkpoint: /home/mabdallah/alexandriax_mt_14d/latentmt_ouro_experiments/ouro26b_latentmt_u4_full87400_ctx3_2shot_r32a64_lr2e4_2ep_bf16_v1/checkpoints/checkpoint-1400


Step,Training Loss
1410,0.984900
1420,0.991900
1430,0.970100
1440,1.007600
1450,1.022100
1460,1.013000
1470,1.001200
1480,0.989100
1490,0.988300
1500,0.916600


In [ ]:
# Select the final adapter (or AXMT_SELECTED_ADAPTER), unload training state, and load one inference model.
from peft import PeftModel
from transformers.trainer_utils import get_last_checkpoint

if SELECTED_ADAPTER_OVERRIDE:
    selected_adapter = Path(SELECTED_ADAPTER_OVERRIDE).expanduser().resolve()
elif (DIRS["final_adapter"] / "adapter_config.json").exists():
    selected_adapter = DIRS["final_adapter"].resolve()
else:
    last = get_last_checkpoint(str(DIRS["checkpoints"]))
    assert last, "No adapter/checkpoint is available."
    selected_adapter = Path(last).resolve()
assert (selected_adapter / "adapter_config.json").exists(), selected_adapter

for name in ("trainer", "model", "train_result"):
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    device_map={"": 0},
)
inference_model = PeftModel.from_pretrained(base_model, selected_adapter, is_trainable=False)
inference_model = inference_model.merge_and_unload(safe_merge=True)
inference_model.config.use_cache = True
inference_model.eval()
tokenizer.padding_side = "left"
atomic_json({"selected_adapter": str(selected_adapter)}, RUN_ROOT / "selected_adapter.json")
print("Selected adapter:", selected_adapter)
print("CUDA allocated GiB:", round(torch.cuda.memory_allocated() / 2**30, 2))


In [ ]:
# Resumable deterministic Beam-4 generation, then official locked-40% metrics.
from tqdm.auto import tqdm
import sacrebleu

GEN_KWARGS = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": 4,
    "do_sample": False,
    "length_penalty": 1.0,
    "repetition_penalty": 1.05,
    "early_stopping": True,
    "use_cache": True,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

def clean_prediction(text):
    text = text.strip()
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[-1].strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    text = text.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
    return text

@torch.inference_mode()
def translate_row(row):
    prompt, prompt_ids = fitted_prompt(row, MAX_SEQ_LEN - MAX_NEW_TOKENS)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(inference_model.device)
    output = inference_model.generate(**inputs, **GEN_KWARGS)
    prediction = clean_prediction(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    if not prediction:
        raise RuntimeError(f"Empty prediction for {row['source_id']}")
    return prediction, len(prompt_ids)

def atomic_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def generation_fingerprint(frame):
    h = hashlib.sha256()
    for sid in frame["source_id"].astype(str):
        h.update(sid.encode()); h.update(b"\n")
    return h.hexdigest()

def generate_resumable(frame, output_csv, label):
    output_csv = Path(output_csv)
    manifest_path = output_csv.with_suffix(".generation.json")
    manifest = {
        "label": label, "adapter": str(selected_adapter), "dataset": generation_fingerprint(frame),
        "rows": len(frame), "generation": GEN_KWARGS,
        "prompt": {"context": MAX_CONTEXT_TURNS, "few_shots": N_FEW_SHOTS, "thinking": False},
    }
    if manifest_path.exists():
        assert json.loads(manifest_path.read_text()) == manifest, (
            f"Cannot resume {label}: adapter/data/generation settings changed."
        )
    else:
        atomic_json(manifest, manifest_path)

    metadata_cols = [
        c for c in ["source_id", "config", "conversation_id", "turn_order",
                    "source_text", "target_arabic"] if c in frame.columns
    ]
    expected_ids = frame["source_id"].astype(str).tolist()
    expected_set, order = set(expected_ids), {sid: i for i, sid in enumerate(expected_ids)}
    records = []
    if output_csv.exists():
        existing = pd.read_csv(output_csv, dtype={"source_id": str, "config": str, "conversation_id": str})
        assert "prediction" in existing and "source_id" in existing
        assert existing["source_id"].is_unique
        assert set(existing["source_id"]).issubset(expected_set)
        records = existing.to_dict("records")
    done = {str(x["source_id"]) for x in records}

    def flush():
        saved = pd.DataFrame(records)
        saved["_order"] = saved["source_id"].map(order)
        saved = saved.sort_values("_order").drop(columns="_order")
        atomic_csv(saved, output_csv)

    new_since_flush = 0
    try:
        for row in tqdm(frame.to_dict("records"), desc=f"{label} Beam-4", total=len(frame)):
            sid = str(row["source_id"])
            if sid in done:
                continue
            prediction, prompt_tokens = translate_row(row)
            record = {c: row[c] for c in metadata_cols}
            record.update({"prediction": prediction, "prompt_tokens": prompt_tokens})
            records.append(record); done.add(sid); new_since_flush += 1
            if new_since_flush >= GENERATION_SAVE_EVERY:
                flush(); new_since_flush = 0
    except BaseException:
        if records:
            flush()
        raise
    flush()
    result = pd.read_csv(output_csv, dtype={"source_id": str, "config": str, "conversation_id": str})
    assert len(result) == len(frame) and set(result["source_id"]) == expected_set
    assert result["prediction"].fillna("").str.strip().ne("").all()
    return result

locked_predictions = generate_resumable(
    locked40, DIRS["locked"] / "locked40_turn_predictions.csv", "locked40"
)

metric_rows = []
for country, group in locked_predictions.groupby("config", sort=True):
    predictions = group["prediction"].astype(str).tolist()
    references = group["target_arabic"].astype(str).tolist()
    metric_rows.append({
        "country": country,
        "turns": len(group),
        "spBLEU": sacrebleu.corpus_bleu(predictions, [references], tokenize="flores200").score,
        "chrF++": sacrebleu.corpus_chrf(predictions, [references], word_order=2).score,
    })
per_country_metrics = pd.DataFrame(metric_rows)
assert per_country_metrics["country"].tolist() == COUNTRY_CODES
macro = {
    "countries": len(per_country_metrics),
    "turns": len(locked_predictions),
    "macro_spBLEU": float(per_country_metrics["spBLEU"].mean()),
    "macro_chrF++": float(per_country_metrics["chrF++"].mean()),
    "adapter": str(selected_adapter),
}
per_country_metrics.to_csv(DIRS["locked"] / "official_per_country_metrics.csv", index=False)
pd.DataFrame([macro]).to_csv(DIRS["locked"] / "official_macro_metrics.csv", index=False)
atomic_json(macro, DIRS["locked"] / "official_metrics.json")
display(per_country_metrics)
display(pd.DataFrame([macro]))


In [ ]:
# Load the already-downloaded private archive, validate its official schema/counts, and infer resumably.
def zip_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def discover_private_archive():
    if PRIVATE_TEST_ARCHIVE:
        candidates = [Path(PRIVATE_TEST_ARCHIVE).expanduser().resolve()]
    else:
        common = [
            PROJECT / "data/alexandriax_private_test.zip",
            PROJECT / "data/nilechat3b_all14/alexandriax_private_test.zip",
        ]
        candidates = [p.resolve() for p in common if p.exists()]
        for root in [PROJECT / "data", PROJECT / "data/nilechat3b_all14"]:
            if root.exists():
                candidates.extend(
                    p.resolve() for p in root.rglob("*.zip")
                    if re.search(r"(private|codabench|test)", p.name, re.I)
                    and "submission" not in p.name.lower()
                )
    candidates = sorted(set(p for p in candidates if p.exists()), key=str)
    valid = []
    for path in candidates:
        try:
            with zipfile.ZipFile(path) as zf:
                jsonls = [n for n in zf.namelist() if n.lower().endswith(".jsonl")
                          and "prediction" not in n.lower()]
            if jsonls:
                valid.append((path, len(jsonls)))
        except zipfile.BadZipFile:
            pass
    if not valid:
        raise FileNotFoundError(
            "The downloaded private-test ZIP was not found. Set AXMT_PRIVATE_TEST_ARCHIVE "
            "to the existing local ZIP; this notebook will not redownload data."
        )
    # Prefer the canonical name, then a 13-JSONL archive, then the shortest path.
    valid.sort(key=lambda x: (
        "alexandriax_private_test" not in x[0].name.lower(),
        x[1] != EXPECTED["countries"], len(str(x[0])), str(x[0])
    ))
    best = valid[0]
    if len(valid) > 1 and valid[1][0] != best[0] and valid[1][1] == best[1]:
        print("Private archive candidates:", [str(x[0]) for x in valid])
        print("Selected by canonical-name/path rule:", best[0])
    return best[0]

archive_path = discover_private_archive()
archive_sha = zip_sha256(archive_path)
extract_dir = DIRS["prepared"] / "private_test_extracted"
extract_manifest = extract_dir / "_archive.json"
expected_extract = {"archive": str(archive_path), "sha256": archive_sha}
if extract_manifest.exists():
    assert json.loads(extract_manifest.read_text()) == expected_extract, (
        "Private archive changed inside this RUN_NAME; use a new RUN_NAME."
    )
else:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as zf:
        for member in zf.infolist():
            if member.is_dir():
                continue
            destination = (extract_dir / member.filename).resolve()
            assert str(destination).startswith(str(extract_dir.resolve()) + os.sep), "Unsafe ZIP member."
            destination.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)
    atomic_json(expected_extract, extract_manifest)

test_records = []
jsonl_files = sorted(
    p for p in extract_dir.rglob("*.jsonl") if "prediction" not in p.name.lower()
)
assert jsonl_files, f"No test JSONL files extracted from {archive_path}"
for jsonl_path in jsonl_files:
    path_country = infer_country_from_path(jsonl_path.relative_to(extract_dir))
    with open(jsonl_path, encoding="utf-8") as fh:
        for line_number, line in enumerate(fh, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            cfg = normalize_country(get_ci(row, ["config", "country", "country_code"], path_country))
            assert cfg in COUNTRY_CODES, f"Unknown country in {jsonl_path}:{line_number}"
            conv = clean_scalar(get_ci(
                row, ["conv_id", "conversation_id", "dialogue_id", "id"], ""
            ))
            assert conv, f"Missing conv_id in {jsonl_path}:{line_number}"
            turns = parse_list(get_ci(
                row, ["english_conversation", "english_turns", "source_turns", "turns"], []
            ))
            assert turns, f"No English turns in {jsonl_path}:{line_number}"
            prior = []
            for i, turn in enumerate(turns):
                source = turn_text(turn)
                assert source, f"Empty source in {cfg}/{conv}/{i+1}"
                order = turn_order(turn, i + 1)
                test_records.append({
                    "source_id": f"{cfg}_test_{conv}_{order}",
                    "config": cfg, "conversation_id": conv, "turn_order": int(order),
                    "source_text": source,
                    "dialect": clean_scalar(get_ci(row, ["dialect", "target_dialect"], "")),
                    "domain": clean_scalar(get_ci(row, ["domain", "topic"], "")),
                    "persona": clean_scalar(get_ci(row, ["persona", "roles"], "")),
                    "speaker": turn_value(turn, ["speaker", "role", "speaker_role", "participant"]),
                    "gender_direction": turn_value(
                        turn, ["direction", "gender_direction", "speaker_addressee_gender"]
                    ),
                    "previous_english_turns": list(prior[-MAX_CONTEXT_TURNS:]),
                    "_origin": "private_test",
                })
                prior.append({
                    "turn_order": int(order),
                    "speaker": turn_value(turn, ["speaker", "role", "speaker_role", "participant"]),
                    "direction": turn_value(
                        turn, ["direction", "gender_direction", "speaker_addressee_gender"]
                    ),
                    "text": source,
                })

private_test = pd.DataFrame(test_records).sort_values(
    ["config", "conversation_id", "turn_order"], kind="stable"
).reset_index(drop=True)
assert len(private_test) == EXPECTED["private_test_turns"], len(private_test)
assert private_test[["config", "conversation_id"]].drop_duplicates().shape[0] == EXPECTED["private_test_conversations"]
assert private_test["source_id"].is_unique
assert sorted(private_test["config"].unique().tolist()) == COUNTRY_CODES
print("Private archive:", archive_path)
print("Private turns/conversations:", len(private_test), private_test[["config","conversation_id"]].drop_duplicates().shape[0])
display(private_test.groupby("config").agg(
    turns=("source_id", "size"), conversations=("conversation_id", "nunique")
))

private_predictions = generate_resumable(
    private_test, DIRS["test"] / "private_test_turn_predictions.csv", "private_test"
)
print("Private inference complete:", len(private_predictions))


In [ ]:
# Build, read back, and cryptographically validate the official submission ZIP.
submission = private_predictions.sort_values(
    ["config", "conversation_id", "turn_order"], kind="stable"
).reset_index(drop=True)
records = []
for (country, conv_id), group in submission.groupby(["config", "conversation_id"], sort=True):
    group = group.sort_values("turn_order", kind="stable")
    assert not group["turn_order"].duplicated().any(), f"Duplicate order in {country}/{conv_id}"
    records.append({
        "conv_id": str(conv_id),
        "country": str(country),
        "turns": [
            {"turn_order": int(row.turn_order), "prediction": str(row.prediction).strip()}
            for row in group.itertuples(index=False)
        ],
    })

assert len(records) == EXPECTED["private_test_conversations"]
assert sum(len(r["turns"]) for r in records) == EXPECTED["private_test_turns"]
predictions_jsonl = DIRS["test"] / "predictions.jsonl"
jsonl_tmp = predictions_jsonl.with_suffix(".jsonl.tmp")
with open(jsonl_tmp, "w", encoding="utf-8") as fh:
    for record in records:
        fh.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")
os.replace(jsonl_tmp, predictions_jsonl)

zip_path = DIRS["test"] / f"{RUN_NAME}_official_submission.zip"
zip_tmp = zip_path.with_suffix(".zip.tmp")
with zipfile.ZipFile(zip_tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(predictions_jsonl, arcname="predictions.jsonl")
os.replace(zip_tmp, zip_path)

expected_keys = set(zip(
    private_test["config"].astype(str),
    private_test["conversation_id"].astype(str),
    private_test["turn_order"].astype(int),
))
readback_keys, readback_conversations = set(), 0
with zipfile.ZipFile(zip_path) as zf:
    assert zf.namelist() == ["predictions.jsonl"], zf.namelist()
    with zf.open("predictions.jsonl") as fh:
        for binary_line in fh:
            record = json.loads(binary_line.decode("utf-8"))
            assert set(record) == {"conv_id", "country", "turns"}
            readback_conversations += 1
            for turn in record["turns"]:
                assert set(turn) == {"turn_order", "prediction"}
                assert str(turn["prediction"]).strip()
                key = (str(record["country"]), str(record["conv_id"]), int(turn["turn_order"]))
                assert key not in readback_keys, f"Duplicate submission key: {key}"
                readback_keys.add(key)
assert readback_conversations == EXPECTED["private_test_conversations"]
assert readback_keys == expected_keys, (
    f"Submission key mismatch: missing={len(expected_keys-readback_keys)}, extra={len(readback_keys-expected_keys)}"
)

submission_manifest = {
    "zip": str(zip_path),
    "sha256": zip_sha256(zip_path),
    "bytes": zip_path.stat().st_size,
    "conversations": readback_conversations,
    "turns": len(readback_keys),
    "countries": COUNTRY_CODES,
    "selected_adapter": str(selected_adapter),
    "locked40_metrics": macro,
    "zip_members": ["predictions.jsonl"],
}
atomic_json(submission_manifest, DIRS["test"] / "submission_manifest.json")
print("READY TO SUBMIT")
print("ZIP:", zip_path)
print("SHA256:", submission_manifest["sha256"])
print("Conversations / turns:", readback_conversations, "/", len(readback_keys))


## Outputs

The last cell prints the exact ready-to-upload ZIP and SHA-256. The run folder also retains:

- `frozen_run_config.json` and resolved data fingerprints;
- resumable checkpoints every 100 optimizer steps;
- the final adapter;
- resumable locked/private turn-level CSVs saved every 100 new predictions;
- per-country and macro official locked-40% metrics;
- the official `predictions.jsonl`, validated ZIP, and submission manifest.

If execution is interrupted, rerun the notebook with the same `RUN_NAME`: tokenization, training, locked inference, and private inference each resume from their last durable state.
